# Manuscript Figure 4

This is a public Colab adaptation of `src/05_mapping/05_figure1_s2_vs_context_comparison_colab.ipynb` from the team snapshot. It uses the original figure and inference cells with paths set to the archived data. Fitted models and cell outputs are not published. Start with a fresh CPU runtime and run from the top.


## Load the team data

Choose the full team ZIP (default), or set `USE_EXISTING_DRIVE_RASTERS = True` to use the small map-support ZIP plus existing Drive folders. Set `ARCHIVE_PATH` if the ZIP is in a subfolder. Both routes use verified inputs and neither signs into Earth Engine. Save generated results to Drive before disconnecting.


In [ ]:
import hashlib
import os
import tempfile
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

USE_EXISTING_DRIVE_RASTERS = False
MY_DRIVE = Path("/content/drive/MyDrive")
ARCHIVE_PATH = MY_DRIVE / (
    "Water_Quality_Map_Support.zip" if USE_EXISTING_DRIVE_RASTERS else "Water_Quality_Processed_Handoff.zip"
)
EXPECTED_SHA256 = "917042102ff89628664e459628e6c7719928e3256a3d17c3e3ca9e51edce480a" if USE_EXISTING_DRIVE_RASTERS else "1f1039b7a6d113685c2c95f840f2a475be5e936e02090bcf5283f2ba9b506773"


HANDOFF_ROOT = Path("/content/wqr-handoff-" + EXPECTED_SHA256[:12])
marker = HANDOFF_ROOT / ".verified-archive-sha256"

if not (marker.is_file() and marker.read_text().strip() == EXPECTED_SHA256):
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            f"{ARCHIVE_PATH} was not found. Ask the project owner for the selected ZIP "
            "and point ARCHIVE_PATH to its location on the mounted Drive."
        )
    with tempfile.TemporaryDirectory(dir=HANDOFF_ROOT.parent) as tmp:
        tmp = Path(tmp)
        local_zip = tmp / ARCHIVE_PATH.name
        digest = hashlib.sha256()
        with ARCHIVE_PATH.open("rb") as src, local_zip.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
                dst.write(chunk)
        if digest.hexdigest() != EXPECTED_SHA256:
            raise ValueError("This ZIP differs from the verified snapshot. Check the filename and download.")
        extracted = tmp / "snapshot"
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(extracted)
        (extracted / marker.name).write_text(EXPECTED_SHA256 + "\n")
        os.replace(extracted, HANDOFF_ROOT)

print("Team data ready:", HANDOFF_ROOT)


In [ ]:
# Only needed for the optional direct-Drive route. Edit paths for Drive shortcuts.
import csv

raster_folders = {'polygon/features': 'Water_Quality_RS_Moving_Window_Polygon'}
expected_prefixes = ['WB02_20240719', 'WB05_20250704', 'WB07_20250120']
DRIVE_RASTER_PATHS = {group: MY_DRIVE / name for group, name in raster_folders.items()}

if USE_EXISTING_DRIVE_RASTERS:
    with (HANDOFF_ROOT / "RASTER_MANIFEST.csv").open(newline="", encoding="utf-8") as handle:
        raster_manifest = list(csv.DictReader(handle))
    for group, drive_folder in DRIVE_RASTER_PATHS.items():
        for prefix in expected_prefixes:
            suffix = "_sentinel2_rgb.tif" if group.endswith("/rgb") else "_moving_window_features.tif"
            relative = f"data_ready/rasters/{group}/{prefix}{suffix}"
            matches = [row for row in raster_manifest if row["path"] == relative]
            if len(matches) != 1:
                raise ValueError(f"No unique reference file for {relative}")
            row = matches[0]
            destination = HANDOFF_ROOT / relative
            if destination.is_file() and destination.stat().st_size == int(row["bytes"]):
                continue  # Only verified files are moved to this runtime-local folder.
            original_raster = drive_folder / destination.name
            if not original_raster.is_file():
                raise FileNotFoundError(f"{original_raster} not found. Check DRIVE_RASTER_PATHS.")
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_suffix(".part")
            try:
                digest = hashlib.sha256()
                with original_raster.open("rb") as src, temporary.open("wb") as dst:
                    for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                        digest.update(chunk)
                        dst.write(chunk)
                if digest.hexdigest() != row["sha256"]:
                    raise ValueError(f"{original_raster.name} differs from the paper snapshot.")
                temporary.replace(destination)
            finally:
                temporary.unlink(missing_ok=True)
    print("Verified selected Drive rasters:", ", ".join(expected_prefixes))


# Figure 1: S2-only versus S2 + Context at three representative water bodies

This Google Colab notebook creates three separate publication figures. Each site has its own execution cell and displays **Turbidity**, **Suspended solids (SS)**, and **DO proxy** as:

1. S2-only prediction
2. S2 + Context prediction
3. Context - S2 prediction difference

The manuscript crops use the exact support-driven logic from Notebook 04. `WB02` uses its complete reviewed-polygon support extent. `WB05` focuses on the largest connected prediction component, while `WB07` removes the broad coastal block and focuses on the same central river bend as the publication map. Cropping changes only the displayed frame; inference, OOD screening, and spatial summaries use the complete prediction raster.

Six locked artifacts are embedded. No fitting or model selection occurs here. The DO result is labelled **DO proxy** because its association with the predictors is indirect.

In [ ]:
# Colab/runtime setup. This cell is also safe to run locally.
from pathlib import Path
import importlib.metadata
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None
required_versions = {
    "scikit-learn": "1.7.2",
    "catboost": "1.2.10",
}
install_specs = []
for package, version in required_versions.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install_specs.append(f"{package}=={version}")

for module_name, package_name in {
    "rasterio": "rasterio",
    "contextily": "contextily",
    "joblib": "joblib",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "geopandas": "geopandas",
    "geopandas": "geopandas",
    "geopandas": "geopandas",
    "geopandas": "geopandas",
    "geopandas": "geopandas",
    "geopandas": "geopandas",
}.items():
    if importlib.util.find_spec(module_name) is None:
        install_specs.append(package_name)

if install_specs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install_specs])

import base64
import hashlib
import json
import math
import re
import warnings

import contextily as ctx
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MaxNLocator
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from IPython.display import display
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import Window
from shapely.geometry import mapping
from scipy.ndimage import label as ndi_label

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Runtime:", "Google Colab" if IN_COLAB else "local")
print("scikit-learn:", importlib.metadata.version("scikit-learn"))
print("catboost:", importlib.metadata.version("catboost"))
print("rasterio:", rasterio.__version__)

In [ ]:
# Fixed representative sites. Publication crops are calculated with
# the same support-driven functions used in Notebook 04.
SITE_CONFIGS = {
    "WB02": {
        "map_id": "WB02",
        "sat_date": "2024-07-19",
        "waterbody": "Thale Luang",
    },
    "WB07": {
        "map_id": "WB07",
        "sat_date": "2025-01-20",
        "waterbody": "Lower Lang Suan River",
    },
    "WB05": {
        "map_id": "WB05",
        "sat_date": "2025-07-04",
        "waterbody": "Pak Phanang River",
    },
}

RIVER_PANEL_ASPECT = 1.60
FEATURE_FILE_OVERRIDE_BY_MAP_ID = {
}
FEATURE_INPUT_ROOT_OVERRIDE = HANDOFF_ROOT / "data_ready/rasters/polygon/features"
feature_root_candidates = [FEATURE_INPUT_ROOT_OVERRIDE]
OUTPUT_ROOT_OVERRIDE = HANDOFF_ROOT / "run_output/colab_fig04"

MAX_OOD_FEATURE_FRACTION = 0.25
BLOCK_SIZE = 512
PREVIEW_MAX_DIMENSION = 1400
NODATA = -9999.0
OVERWRITE_PREDICTION = False
USE_SATELLITE_BASEMAP = True
BASEMAP_SOURCE = ctx.providers.Esri.WorldImagery
BASEMAP_ZOOM = "auto"
BASEMAP_WHITE_VEIL_ALPHA = 0.52
PREDICTION_VIEW_PADDING_FRACTION = 0.12
PREDICTION_VIEW_MIN_PADDING_PIXELS = 2

OUTPUT_ROOT = OUTPUT_ROOT_OVERRIDE
RASTER_DIR = OUTPUT_ROOT / "geotiff"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
for directory in [RASTER_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Notebook 04 used this reviewed geometry before calculating map bounds.
# It is required here so the exact same polygon-part support is retained.
POLYGON_PATH = HANDOFF_ROOT / "data_ready/maps/polygons/waterbody_polygons.geojson"
if not POLYGON_PATH.exists():
    raise FileNotFoundError(
        f"Missing reviewed Notebook 04 polygons: {POLYGON_PATH}"
    )
waterbody_polygons = gpd.read_file(POLYGON_PATH)
waterbody_polygons["map_id"] = waterbody_polygons["map_id"].astype(str)
if waterbody_polygons.crs is None:
    waterbody_polygons = waterbody_polygons.set_crs("EPSG:4326")
if waterbody_polygons["map_id"].duplicated().any():
    raise ValueError("Expected one reviewed Polygon/MultiPolygon per map_id.")

print("Output root:", OUTPUT_ROOT)
print("Reviewed polygons:", POLYGON_PATH)
display(pd.DataFrame(SITE_CONFIGS).T)

In [ ]:
# Load the exact historical setup cell from the verified team snapshot.
import hashlib
import json

original = HANDOFF_ROOT / "src/05_mapping/05_figure1_s2_vs_context_comparison_colab.ipynb"
private_notebook = json.loads(original.read_text(encoding="utf-8-sig"))
private_setup = "".join(private_notebook["cells"][3]["source"])
if hashlib.sha256(private_setup.encode()).hexdigest() != "4ba78717c825c99ad9a25ee378a610364fd8431f4315fb624e1e6482b34e52b1":
    raise ValueError("The locked-model setup cell does not match the team snapshot.")
exec(compile(private_setup, str(original), "exec"))
del private_setup, private_notebook


In [ ]:
# Load the six locked model bundles once; all three site cells reuse them.
bundles = {}
artifact_file_by_id = {}
inventory_rows = []

for file_name in EMBEDDED_ARTIFACTS:
    bundle = joblib.load(ARTIFACT_ROOT / file_name)
    artifact_id = bundle["artifact_id"]
    bundles[artifact_id] = bundle
    artifact_file_by_id[artifact_id] = file_name

EXPECTED_ARTIFACT_IDS = [
    "turbidity_s2", "turbidity_s2_context",
    "ss_s2", "ss_s2_context",
    "do_s2", "do_s2_context",
]
if set(bundles) != set(EXPECTED_ARTIFACT_IDS):
    raise RuntimeError(f"Unexpected artifact IDs: {sorted(bundles)}")

for artifact_id in EXPECTED_ARTIFACT_IDS:
    bundle = bundles[artifact_id]
    metrics = bundle["validation_metrics"]
    model_name = (
        "LogisticRegression + LinearSVR (soft gate)"
        if bundle["prediction_contract"]["strategy"] == "two_stage_soft_gate"
        else bundle.get(
            "controlled_model_family",
            type(bundle["estimator"].named_steps["model"]).__name__,
        )
    )
    if model_name == "CatBoostRegressor":
        model_name = "CatBoost"
    inventory_rows.append({
        "artifact_id": artifact_id,
        "target": bundle["target"],
        "feature_set": bundle["feature_set"],
        "model": model_name,
        "n_features": len(bundle["feature_names"]),
        "n_train": bundle["n_train"],
        "n_test": bundle["n_test"],
        "r2": metrics.get("original_r2", metrics.get("r2")),
        "rmse": metrics.get("original_rmse", metrics.get("rmse")),
        "mae": metrics.get("original_mae", metrics.get("mae")),
    })

model_inventory = pd.DataFrame(inventory_rows)
model_inventory.to_csv(
    TABLE_DIR / "figure1_model_inventory.csv", index=False, encoding="utf-8-sig"
)
display(model_inventory.round(4))

In [ ]:
# Chunked inference helpers. Predictions stay in original target units.
OUTPUT_BANDS = [
    "prediction_support",
    "turbidity_s2", "turbidity_s2_context", "turbidity_context_minus_s2",
    "ss_s2", "ss_s2_context", "ss_context_minus_s2",
    "do_s2", "do_s2_context", "do_context_minus_s2",
    "turbidity_s2_ood_fraction", "turbidity_s2_context_ood_fraction",
    "ss_s2_ood_fraction", "ss_s2_context_ood_fraction",
    "do_s2_ood_fraction", "do_s2_context_ood_fraction",
]
OUTPUT_BAND_INDEX = {name: index + 1 for index, name in enumerate(OUTPUT_BANDS)}

range_lookup = {}
for artifact_id, group in training_ranges.groupby("artifact_id"):
    indexed = group.set_index("feature_name")
    feature_names = bundles[artifact_id]["feature_names"]
    range_lookup[artifact_id] = {
        "minimum": indexed.loc[feature_names, "training_min"].to_numpy(float),
        "maximum": indexed.loc[feature_names, "training_max"].to_numpy(float),
    }

def predict_bundle(bundle, feature_frame):
    strategy = bundle["prediction_contract"]["strategy"]
    if strategy == "log1p_regression":
        predicted_log = np.asarray(bundle["estimator"].predict(feature_frame), dtype=float)
        prediction = np.expm1(np.clip(predicted_log, 0.0, None))
    elif strategy == "raw_regression":
        prediction = np.asarray(bundle["estimator"].predict(feature_frame), dtype=float)
    elif strategy == "two_stage_soft_gate":
        classifier = bundle["classifier"]
        probabilities = np.asarray(classifier.predict_proba(feature_frame), dtype=float)
        classes = np.asarray(classifier.classes_)
        high_columns = np.flatnonzero(classes == 1)
        if high_columns.size != 1:
            raise ValueError(f"Could not identify high-class probability for {bundle['artifact_id']}")
        p_high = probabilities[:, high_columns[0]]
        predicted_log = np.asarray(bundle["regressor"].predict(feature_frame), dtype=float)
        high_prediction = np.expm1(np.clip(predicted_log, 0.0, None))
        prediction = p_high * high_prediction + (1.0 - p_high) * float(bundle["low_value"])
    else:
        raise ValueError(f"Unsupported strategy: {strategy}")
    return np.clip(prediction, 0.0, None)

def ood_fraction(artifact_id, matrix):
    bounds = range_lookup[artifact_id]
    finite = np.isfinite(matrix)
    outside = finite & (
        (matrix < bounds["minimum"][None, :])
        | (matrix > bounds["maximum"][None, :])
    )
    # Missing values are handled by each locked pipeline and are not labelled out-of-range.
    return outside.sum(axis=1) / matrix.shape[1]

def iter_windows(width, height, block_size):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(
                col_off,
                row_off,
                min(block_size, width - col_off),
                min(block_size, height - row_off),
            )

def model_features_from_block(block_by_name, feature_names, selected_flat):
    matrix = np.column_stack([
        block_by_name[name].reshape(-1)[selected_flat]
        for name in feature_names
    ])
    return matrix, pd.DataFrame(matrix, columns=feature_names)

In [ ]:
# Multi-site inference, preview, crop, and plotting functions.
def resolve_feature_path(site):
    map_id = site["map_id"]
    override = FEATURE_FILE_OVERRIDE_BY_MAP_ID.get(map_id)
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    roots = (
        [Path(FEATURE_INPUT_ROOT_OVERRIDE)]
        if FEATURE_INPUT_ROOT_OVERRIDE
        else [path for path in feature_root_candidates if path.exists()]
    )
    if not roots:
        raise FileNotFoundError(
            "No moving-window feature directory was found. Set FEATURE_INPUT_ROOT_OVERRIDE."
        )
    date_token = pd.Timestamp(site["sat_date"]).strftime("%Y%m%d")
    prefix = f"{map_id}_{date_token}_moving_window_features"
    matches = []
    for root in roots:
        matches.extend(
            path for path in root.rglob(f"{prefix}*.tif")
            if "water_quality_predictions" not in path.name
        )
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one logical feature GeoTIFF for {map_id} on {site['sat_date']}; "
            f"found {len(matches)}:\n" + "\n".join(str(path) for path in matches)
        )
    return matches[0]


def verify_feature_raster(path):
    with rasterio.open(path) as source:
        band_names = list(source.descriptions)
        if source.count != 62 or len(set(band_names)) != 62:
            raise ValueError(f"Expected 62 unique Notebook 03 bands: {path}")
        for artifact_id in EXPECTED_ARTIFACT_IDS:
            missing = [
                feature for feature in bundles[artifact_id]["feature_names"]
                if feature not in band_names
            ]
            if missing:
                raise KeyError(f"{artifact_id} missing raster features: {missing}")


def prediction_path_for_site(site):
    date_token = pd.Timestamp(site["sat_date"]).strftime("%Y%m%d")
    return RASTER_DIR / f"{site['map_id']}_{date_token}_s2_vs_context_predictions.tif"


def run_locked_prediction(site):
    feature_path = resolve_feature_path(site)
    verify_feature_raster(feature_path)
    output_path = prediction_path_for_site(site)
    if output_path.exists() and not OVERWRITE_PREDICTION:
        print("Using existing prediction raster:", output_path)
        return feature_path, output_path

    with rasterio.open(feature_path) as source:
        band_index_by_name = {
            name: index + 1 for index, name in enumerate(source.descriptions)
        }
        union_features = []
        for artifact_id in EXPECTED_ARTIFACT_IDS:
            for feature in bundles[artifact_id]["feature_names"]:
                if feature not in union_features:
                    union_features.append(feature)
        read_names = union_features + ["prediction_support"]
        read_indexes = [band_index_by_name[name] for name in read_names]

        profile = source.profile.copy()
        profile.update(
            driver="GTiff", count=len(OUTPUT_BANDS), dtype="float32",
            nodata=NODATA, compress="deflate", predictor=3, BIGTIFF="IF_SAFER",
        )
        with rasterio.open(output_path, "w", **profile) as destination:
            for band_index, band_name in enumerate(OUTPUT_BANDS, start=1):
                destination.set_band_description(band_index, band_name)
            windows = list(iter_windows(source.width, source.height, BLOCK_SIZE))
            for window_number, window in enumerate(windows, start=1):
                raw = source.read(
                    read_indexes, window=window, masked=True
                ).filled(np.nan).astype(float)
                block_by_name = {
                    name: raw[index] for index, name in enumerate(read_names)
                }
                support = (
                    np.isfinite(block_by_name["prediction_support"])
                    & (block_by_name["prediction_support"] > 0)
                )
                selected_flat = np.flatnonzero(support.reshape(-1))
                output = np.full(
                    (len(OUTPUT_BANDS), int(window.height), int(window.width)),
                    NODATA, dtype=np.float32,
                )
                output[0] = np.where(support, 1.0, NODATA)

                predictions = {}
                ood_values = {}
                if selected_flat.size:
                    for artifact_id in EXPECTED_ARTIFACT_IDS:
                        bundle = bundles[artifact_id]
                        matrix, feature_frame = model_features_from_block(
                            block_by_name, bundle["feature_names"], selected_flat
                        )
                        predictions[artifact_id] = predict_bundle(bundle, feature_frame)
                        ood_values[artifact_id] = ood_fraction(artifact_id, matrix)

                    output_values = {
                        "turbidity_s2": predictions["turbidity_s2"],
                        "turbidity_s2_context": predictions["turbidity_s2_context"],
                        "turbidity_context_minus_s2": predictions["turbidity_s2_context"] - predictions["turbidity_s2"],
                        "ss_s2": predictions["ss_s2"],
                        "ss_s2_context": predictions["ss_s2_context"],
                        "ss_context_minus_s2": predictions["ss_s2_context"] - predictions["ss_s2"],
                        "do_s2": predictions["do_s2"],
                        "do_s2_context": predictions["do_s2_context"],
                        "do_context_minus_s2": predictions["do_s2_context"] - predictions["do_s2"],
                    }
                    for artifact_id in EXPECTED_ARTIFACT_IDS:
                        output_values[f"{artifact_id}_ood_fraction"] = ood_values[artifact_id]
                    for output_name, values in output_values.items():
                        flat = output[OUTPUT_BAND_INDEX[output_name] - 1].reshape(-1)
                        flat[selected_flat] = values.astype(np.float32)
                destination.write(output, window=window)
                if window_number == 1 or window_number % 10 == 0 or window_number == len(windows):
                    print(f"{site['map_id']}: window {window_number}/{len(windows)}")
    print("Saved prediction raster:", output_path)
    return feature_path, output_path


def read_preview(source, band_name, resampling=Resampling.bilinear):
    scale = min(1.0, PREVIEW_MAX_DIMENSION / max(source.width, source.height))
    width = max(1, int(round(source.width * scale)))
    height = max(1, int(round(source.height * scale)))
    values = source.read(
        OUTPUT_BAND_INDEX[band_name], out_shape=(height, width),
        masked=True, resampling=resampling,
    ).filled(np.nan).astype(float)
    transform = source.transform * Affine.scale(
        source.width / width, source.height / height
    )
    return values, transform


TARGET_SPECS = [
    {"key": "turbidity", "label": "Turbidity", "unit": "NTU", "s2_id": "turbidity_s2", "context_id": "turbidity_s2_context", "difference_limit": 1.0, "difference_ticks": [-1.0, 0.0, 1.0]},
    {"key": "ss", "label": "Suspended solids", "unit": "mg/L", "s2_id": "ss_s2", "context_id": "ss_s2_context", "difference_limit": None, "difference_ticks": None},
    {"key": "do", "label": "DO proxy", "unit": "mg/L", "s2_id": "do_s2", "context_id": "do_s2_context", "difference_limit": 2.0, "difference_ticks": [-2.0, 0.0, 2.0]},
]


def build_site_preview(site, prediction_path):
    preview_data = {}
    summary_rows = []
    with rasterio.open(prediction_path) as source:
        source_crs = source.crs
        support, transform = read_preview(
            source, "prediction_support", Resampling.nearest
        )
        geometry_rows = waterbody_polygons.loc[
            waterbody_polygons["map_id"].eq(site["map_id"])
        ]
        if len(geometry_rows) != 1:
            raise ValueError(
                f"Expected one reviewed polygon for {site['map_id']}; "
                f"found {len(geometry_rows)}"
            )
        geometry = geometry_rows.to_crs(source.crs).geometry.iloc[0]
        polygon_mask = geometry_mask(
            [mapping(geometry)],
            out_shape=support.shape,
            transform=transform,
            invert=True,
            all_touched=True,
        )
        display_support = (
            np.isfinite(support) & (support > 0) & polygon_mask
        )
        # Notebook 04 publication masking: WB05 contains several
        # disconnected water detections. Retain only the largest
        # connected river-mouth component for display and summaries.
        if site["map_id"] == "WB05" and display_support.any():
            labels, count = ndi_label(
                display_support,
                structure=np.ones((3, 3), dtype=np.uint8),
            )
            if count > 0:
                sizes = np.bincount(labels.ravel())
                sizes[0] = 0
                display_support = labels == sizes.argmax()
        if not display_support.any():
            raise RuntimeError(
                f"No prediction-support pixels inside reviewed polygon: {site['map_id']}"
            )
        for spec in TARGET_SPECS:
            s2, _ = read_preview(source, spec["s2_id"])
            context, _ = read_preview(source, spec["context_id"])
            difference, _ = read_preview(source, f"{spec['key']}_context_minus_s2")
            s2_ood, _ = read_preview(
                source, f"{spec['s2_id']}_ood_fraction", Resampling.nearest
            )
            context_ood, _ = read_preview(
                source, f"{spec['context_id']}_ood_fraction", Resampling.nearest
            )
            common_mask = (
                display_support
                & np.isfinite(s2) & np.isfinite(context) & np.isfinite(difference)
                & np.isfinite(s2_ood) & np.isfinite(context_ood)
                & (s2_ood <= MAX_OOD_FEATURE_FRACTION)
                & (context_ood <= MAX_OOD_FEATURE_FRACTION)
            )
            if not common_mask.any():
                raise RuntimeError(f"No common low-OOD pixels: {site['map_id']} / {spec['label']}")
            s2 = np.where(common_mask, s2, np.nan)
            context = np.where(common_mask, context, np.nan)
            difference = np.where(common_mask, difference, np.nan)
            preview_data[spec["key"]] = {
                "s2": s2, "context": context,
                "difference": difference, "mask": common_mask,
            }
            delta = difference[common_mask]
            summary_rows.append({
                "target": spec["label"], "unit": spec["unit"],
                "map_id": site["map_id"], "waterbody": site["waterbody"],
                "sat_date": site["sat_date"],
                "n_common_low_ood_pixels_preview": int(common_mask.sum()),
                "s2_median": float(np.nanmedian(s2)),
                "context_median": float(np.nanmedian(context)),
                "difference_mean": float(np.mean(delta)),
                "difference_median": float(np.median(delta)),
                "difference_p05": float(np.quantile(delta, 0.05)),
                "difference_p95": float(np.quantile(delta, 0.95)),
                "fraction_context_higher": float(np.mean(delta > 0)),
            })
    return (
        preview_data,
        pd.DataFrame(summary_rows),
        transform,
        source_crs,
        display_support,
    )


_SITE_PREVIEW_CACHE = {}
_SHARED_DIFFERENCE_SCALE_CACHE = None


def load_site_preview(site):
    # Load or create one prediction raster, then cache its preview.
    cache_key = (site["map_id"], site["sat_date"])
    if cache_key not in _SITE_PREVIEW_CACHE:
        feature_path, prediction_path = run_locked_prediction(site)
        preview = build_site_preview(site, prediction_path)
        _SITE_PREVIEW_CACHE[cache_key] = {
            "feature_path": feature_path,
            "prediction_path": prediction_path,
            "preview": preview,
        }
    return _SITE_PREVIEW_CACHE[cache_key]


def shared_difference_scales():
    # Use fixed Turbidity/DO limits and pooled P99 for SS.
    global _SHARED_DIFFERENCE_SCALE_CACHE
    if _SHARED_DIFFERENCE_SCALE_CACHE is not None:
        return _SHARED_DIFFERENCE_SCALE_CACHE

    ss_chunks = []
    for site in SITE_CONFIGS.values():
        preview_data = load_site_preview(site)["preview"][0]
        values = np.asarray(
            preview_data["ss"]["difference"], dtype=float
        )
        values = np.abs(values[np.isfinite(values)])
        if values.size:
            ss_chunks.append(values)
    if not ss_chunks:
        raise RuntimeError("No pooled SS differences were available.")

    pooled_ss = np.concatenate(ss_chunks)
    ss_limit = float(np.quantile(pooled_ss, 0.99))
    ss_limit = max(ss_limit, 1e-6)
    _SHARED_DIFFERENCE_SCALE_CACHE = {
        "turbidity": {
            "limit": 1.0, "ticks": [-1.0, 0.0, 1.0],
            "method": "fixed manuscript scale",
        },
        "ss": {
            "limit": ss_limit,
            "ticks": [-ss_limit, 0.0, ss_limit],
            "method": "pooled P99 absolute pixelwise difference",
        },
        "do": {
            "limit": 2.0, "ticks": [-2.0, 0.0, 2.0],
            "method": "fixed manuscript scale",
        },
    }
    scale_table = pd.DataFrame([
        {
            "target": spec["label"],
            "unit": spec["unit"],
            "difference_limit": _SHARED_DIFFERENCE_SCALE_CACHE[spec["key"]]["limit"],
            "method": _SHARED_DIFFERENCE_SCALE_CACHE[spec["key"]]["method"],
        }
        for spec in TARGET_SPECS
    ])
    scale_table.to_csv(
        TABLE_DIR / "figure1_shared_difference_scales.csv",
        index=False, encoding="utf-8-sig",
    )
    display(scale_table.round(4))
    return _SHARED_DIFFERENCE_SCALE_CACHE


# Truncated cividis keeps perceptual ordering while avoiding the
# visually dominant bright-yellow endpoint.
_cividis = plt.get_cmap("cividis")
WATER_CMAP = LinearSegmentedColormap.from_list(
    "cividis_truncated", _cividis(np.linspace(0.05, 0.90, 256))
)
DIFFERENCE_CMAP = LinearSegmentedColormap.from_list(
    "context_difference", ["#2166AC", "#F0F0F0", "#B2182B"]
)
WATER_CMAP.set_bad(alpha=0)
DIFFERENCE_CMAP.set_bad(alpha=0)


def raster_extent(transform, shape):
    height, width = shape
    corners = [
        transform * (0, 0), transform * (width, 0),
        transform * (0, height), transform * (width, height),
    ]
    xs = [point[0] for point in corners]
    ys = [point[1] for point in corners]
    return min(xs), max(xs), min(ys), max(ys)


def prediction_support_bounds(mask, transform):
    rows, cols = np.where(mask)
    row_min, row_max = int(rows.min()), int(rows.max()) + 1
    col_min, col_max = int(cols.min()), int(cols.max()) + 1
    corners = [
        transform * (col_min, row_min), transform * (col_max, row_min),
        transform * (col_min, row_max), transform * (col_max, row_max),
    ]
    xs = [point[0] for point in corners]
    ys = [point[1] for point in corners]
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    pixel_size = max(abs(transform.a), abs(transform.e))
    padding = max(
        (right - left) * PREDICTION_VIEW_PADDING_FRACTION,
        (top - bottom) * PREDICTION_VIEW_PADDING_FRACTION,
        pixel_size * PREDICTION_VIEW_MIN_PADDING_PIXELS,
    )
    return left - padding, bottom - padding, right + padding, top + padding


RIVER_MAP_IDS = {"WB07", "WB05"}
WB07_FOCUS_X_FRACTION = (0.92, 0.95)
AUTO_ZOOM_PAD_FRAC = {"WB05": 0.03, "WB07": 0.03}


def publication_support(site, display_support):
    # Match Notebook 04: isolate WB05's largest connected component.
    support = np.asarray(display_support, dtype=bool)
    if site["map_id"] != "WB05":
        return support
    labels, count = ndi_label(
        support, structure=np.ones((3, 3), dtype=np.uint8)
    )
    if count == 0:
        return support
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == sizes.argmax()


def crop_support(site, display_support):
    # Match Notebook 04's crop-selection support exactly.
    support = publication_support(site, display_support)
    if site["map_id"] != "WB07" or not support.any():
        return support

    n_rows, _n_cols = support.shape
    occupied_cols = np.flatnonzero(support.any(axis=0))
    if occupied_cols.size < 4:
        return support

    support_span = int(occupied_cols[-1] - occupied_cols[0] + 1)
    dense_cols = support.sum(axis=0) >= max(
        3, int(round(0.20 * n_rows))
    )
    dense_labels, dense_count = ndi_label(dense_cols.astype(np.uint8))
    candidates = []
    min_run_width = max(3, int(round(0.04 * support_span)))
    right_tolerance = max(2, int(round(0.03 * support_span)))

    for label_id in range(1, dense_count + 1):
        run_cols = np.flatnonzero(dense_labels == label_id)
        if run_cols.size < min_run_width:
            continue
        if run_cols[-1] < occupied_cols[-1] - right_tolerance:
            continue
        candidates.append(run_cols)

    if not candidates:
        return support

    coastal_run = max(candidates, key=lambda cols: cols[-1])
    mouth_buffer = max(2, int(round(0.025 * support_span)))
    cutoff_col = min(int(coastal_run[0]) + mouth_buffer, support.shape[1])
    focused = support.copy()
    focused[:, cutoff_col:] = False
    if not focused.any():
        return support

    river_cols = np.flatnonzero(focused.any(axis=0))
    river_left = int(river_cols[0])
    river_span = int(river_cols[-1] - river_cols[0] + 1)
    focus_start, focus_end = WB07_FOCUS_X_FRACTION
    focus_col_lo = river_left + int(round(focus_start * river_span))
    focus_col_hi = river_left + int(round(focus_end * river_span))
    focus_col_hi = max(focus_col_hi, focus_col_lo + 1)
    focused[:, :focus_col_lo] = False
    focused[:, focus_col_hi:] = False
    return focused if focused.any() else support


def expand_bounds_to_aspect(bounds, target_aspect):
    left, bottom, right, top = bounds
    width = max(right - left, 1e-9)
    height = max(top - bottom, 1e-9)
    center_x, center_y = (left + right) / 2, (bottom + top) / 2
    if width / height > target_aspect:
        height = width / target_aspect
    else:
        width = height * target_aspect
    return (
        center_x - width / 2, center_y - height / 2,
        center_x + width / 2, center_y + height / 2,
    )


def support_mask_bounds(mask, transform, pad_frac):
    # Map-coordinate bounds with independent x/y padding (Notebook 04).
    rows, cols = np.where(mask)
    if rows.size == 0:
        return None
    row_lo, row_hi = int(rows.min()), int(rows.max()) + 1
    col_lo, col_hi = int(cols.min()), int(cols.max()) + 1
    corners = [
        transform * (col_lo, row_lo),
        transform * (col_hi, row_lo),
        transform * (col_lo, row_hi),
        transform * (col_hi, row_hi),
    ]
    xs = [point[0] for point in corners]
    ys = [point[1] for point in corners]
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    width = max(right - left, 1e-9)
    height = max(top - bottom, 1e-9)
    return (
        left - width * pad_frac,
        bottom - height * pad_frac,
        right + width * pad_frac,
        top + height * pad_frac,
    )


def publication_bounds(site, display_support, transform):
    # Exact Notebook 04 order: target-specific crop support -> 3% pad
    # for river sites -> expand (never contract) to a 1.60 aspect.
    if site["map_id"] in RIVER_MAP_IDS:
        bounds = support_mask_bounds(
            crop_support(site, display_support),
            transform,
            AUTO_ZOOM_PAD_FRAC[site["map_id"]],
        )
        if bounds is None:
            raise RuntimeError(f"No crop support for {site['map_id']}")
        bounds = expand_bounds_to_aspect(bounds, RIVER_PANEL_ASPECT)
        return bounds
    return prediction_support_bounds(display_support, transform)


def add_background(axis, bounds, crs):
    left, bottom, right, top = bounds
    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)
    axis.set_facecolor("#D9DEDC")
    if USE_SATELLITE_BASEMAP:
        try:
            ctx.add_basemap(
                axis, crs=crs, source=BASEMAP_SOURCE, zoom=BASEMAP_ZOOM,
                attribution=False, reset_extent=False, zorder=0,
            )
        except Exception as error:
            print("Basemap warning:", error)
    axis.add_patch(Rectangle(
        (left, bottom), right - left, top - bottom,
        facecolor="white", edgecolor="none",
        alpha=BASEMAP_WHITE_VEIL_ALPHA, zorder=1,
    ))


def add_north_arrow(axis):
    axis.annotate(
        "N", xy=(0.94, 0.91), xytext=(0.94, 0.79),
        xycoords="axes fraction", textcoords="axes fraction",
        ha="center", va="center", fontsize=7, fontweight="bold",
        arrowprops={"arrowstyle": "-|>", "linewidth": 0.9, "color": "#202020"},
        zorder=7,
    )


def add_scale_bar(axis, crs, bounds):
    left, bottom, right, top = bounds
    if str(crs).upper() == "EPSG:4326":
        latitude = (bottom + top) / 2
        width_km = (right - left) * 111.32 * math.cos(math.radians(latitude))
    else:
        width_km = (right - left) / 1000.0
    candidates = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100])
    bar_km = float(candidates[np.argmin(np.abs(candidates - width_km * 0.22))])
    bar_fraction = bar_km / max(width_km, 1e-9)
    x0, y0 = 0.07, 0.07
    axis.plot([x0, x0 + bar_fraction], [y0, y0], transform=axis.transAxes,
              color="black", lw=2.2, zorder=7)
    for x in [x0, x0 + bar_fraction]:
        axis.plot([x, x], [y0 - 0.012, y0 + 0.012], transform=axis.transAxes,
                  color="black", lw=1, zorder=7)
    axis.text(
        x0 + bar_fraction / 2, y0 + 0.018, f"{bar_km:g} km",
        transform=axis.transAxes, ha="center", va="bottom", fontsize=7,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=1.0), zorder=7,
    )


def create_site_figure(site):
    cached = load_site_preview(site)
    feature_path = cached["feature_path"]
    prediction_path = cached["prediction_path"]
    (
        preview_data, summary, transform, crs, display_support
    ) = cached["preview"]
    difference_scales = shared_difference_scales()
    bounds = publication_bounds(site, display_support, transform)
    shape = preview_data["turbidity"]["mask"].shape
    extent = raster_extent(transform, shape)
    origin = "upper" if transform.e < 0 else "lower"
    figure_stem = (
        f"figure1_s2_vs_context_{site['map_id']}_"
        f"{pd.Timestamp(site['sat_date']).strftime('%Y%m%d')}"
    )

    figure_width = 9.2 if site["map_id"] == "WB02" else 12.2
    figure = plt.figure(figsize=(figure_width, 9.3), constrained_layout=False)
    grid = GridSpec(
        9, 3, figure=figure,
        height_ratios=[1.0, 0.045, 0.08, 1.0, 0.045, 0.08, 1.0, 0.045, 0.03],
        hspace=0.05, wspace=0.08,
        left=0.115, right=0.985, top=0.91, bottom=0.045,
    )
    date_value = pd.Timestamp(site["sat_date"])
    date_label = f"{date_value.day} {date_value.strftime('%B %Y')}"
    figure.text(
        0.115, 0.965,
        f"{site['waterbody']} ({site['map_id']}), {date_label}",
        ha="left", va="center", fontsize=12, fontweight="bold",
        color="#222222",
    )

    column_titles = ["S2 only", "S2 + context", "S2 + context − S2 only"]
    summary_indexed = summary.set_index("target")
    for row_index, spec in enumerate(TARGET_SPECS):
        data = preview_data[spec["key"]]
        combined = np.concatenate([
            data["s2"][np.isfinite(data["s2"])],
            data["context"][np.isfinite(data["context"])],
        ])
        prediction_norm = Normalize(
            vmin=max(0.0, float(combined.min())),
            vmax=max(float(combined.max()), float(combined.min()) + 1e-9),
            clip=True,
        )
        difference_scale = difference_scales[spec["key"]]
        difference_limit = float(difference_scale["limit"])
        difference_norm = TwoSlopeNorm(
            vmin=-difference_limit, vcenter=0.0, vmax=difference_limit
        )
        map_row = row_index * 3
        bar_row = map_row + 1
        axes = [figure.add_subplot(grid[map_row, column]) for column in range(3)]
        for column_index, axis in enumerate(axes):
            add_background(axis, bounds, crs)
            values = [data["s2"], data["context"], data["difference"]][column_index]
            axis.imshow(
                values, extent=extent, origin=origin,
                cmap=WATER_CMAP if column_index < 2 else DIFFERENCE_CMAP,
                norm=prediction_norm if column_index < 2 else difference_norm,
                interpolation="nearest", alpha=0.98, zorder=3,
            )
            common_mask = np.asarray(data["mask"], dtype=bool)
            if common_mask.any() and (~common_mask).any():
                axis.contour(
                    common_mask.astype(float), levels=[0.5],
                    extent=extent, origin=origin,
                    colors=["#4A4A4A"], linewidths=0.35, zorder=4,
                )
            axis.set_xlim(bounds[0], bounds[2])
            axis.set_ylim(bounds[1], bounds[3])
            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_color("#555555")
                spine.set_linewidth(0.6)
            if row_index == 0:
                axis.set_title(
                    column_titles[column_index], fontsize=10.5,
                    fontweight="bold", pad=6,
                )
            if column_index == 0:
                axis.text(
                    -0.12, 0.5,
                    "Suspended\nsolids" if spec["key"] == "ss" else spec["label"],
                    transform=axis.transAxes, rotation=0,
                    ha="right", va="center", fontsize=9.5,
                    fontweight="bold", linespacing=1.0,
                )
            panel_letter = chr(ord("a") + row_index * 3 + column_index)
            axis.text(
                0.015, 0.975, f"({panel_letter})",
                transform=axis.transAxes, ha="left", va="top",
                fontsize=8.5, fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=1.0),
                zorder=8,
            )
            if row_index == 2 and column_index == 0:
                add_scale_bar(axis, crs, bounds)
            if row_index == 0 and column_index == 2:
                add_north_arrow(axis)

        prediction_bar_grid = grid[bar_row, 0:2].subgridspec(
            1, 3, width_ratios=[0.12, 0.76, 0.12], wspace=0
        )
        prediction_cax = figure.add_subplot(prediction_bar_grid[0, 1])
        prediction_cbar = figure.colorbar(
            ScalarMappable(norm=prediction_norm, cmap=WATER_CMAP),
            cax=prediction_cax, orientation="horizontal",
        )
        prediction_cbar.locator = MaxNLocator(nbins=4, min_n_ticks=3)
        prediction_cbar.update_ticks()
        prediction_cbar.set_label(
            f"{spec['label']} ({spec['unit']})",
            fontsize=8.2, labelpad=2,
        )
        prediction_cbar.ax.tick_params(labelsize=7.5, length=2.5, width=0.5, pad=1)
        prediction_cbar.outline.set_linewidth(0.45)

        difference_bar_grid = grid[bar_row, 2].subgridspec(
            1, 3, width_ratios=[0.05, 0.90, 0.05], wspace=0
        )
        difference_cax = figure.add_subplot(difference_bar_grid[0, 1])
        difference_cbar = figure.colorbar(
            ScalarMappable(norm=difference_norm, cmap=DIFFERENCE_CMAP),
            cax=difference_cax, orientation="horizontal",
            ticks=difference_scale["ticks"],
            extend="both", extendfrac=0.035,
        )
        difference_cbar.set_label(
            f"Δ {spec['label']} ({spec['unit']})",
            fontsize=8.2, labelpad=3,
        )
        difference_cbar.ax.tick_params(labelsize=7.5, length=2.5, width=0.5, pad=1)
        difference_cbar.ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda value, _position: (
                    "0" if abs(value) < 1e-12 else f"{value:+.2g}"
                )
            )
        )
        difference_cbar.outline.set_linewidth(0.45)
        delta = summary_indexed.loc[spec["label"]]
        difference_cax.text(
            0.5, 1.55,
            f"Median Δ = {delta['difference_median']:+.2f} {spec['unit']}",
            transform=difference_cax.transAxes,
            ha="center", va="bottom", fontsize=7.8, color="#222222",
            clip_on=False,
        )

    paths = {
        "png": FIGURE_DIR / f"{figure_stem}.png",
        "pdf": FIGURE_DIR / f"{figure_stem}.pdf",
        "tif": FIGURE_DIR / f"{figure_stem}.tif",
    }
    figure.savefig(paths["png"], dpi=600, bbox_inches="tight", facecolor="white")
    figure.savefig(paths["pdf"], bbox_inches="tight", facecolor="white")
    figure.savefig(
        paths["tif"], dpi=600, bbox_inches="tight", facecolor="white",
        pil_kwargs={"compression": "tiff_lzw"},
    )
    summary.to_csv(
        TABLE_DIR / f"{figure_stem}_spatial_summary.csv",
        index=False, encoding="utf-8-sig",
    )
    plt.show()
    display(summary.round(4))
    print("Feature:", feature_path)
    print("Prediction:", prediction_path)
    print("Figure:", paths["png"])
    return {"site": site, "prediction_path": prediction_path, "summary": summary, **paths}

## A. Thale Luang (WB02; 2024-07-19)

In [ ]:
# Cell A: complete WB02 prediction-support extent.
result_WB02 = create_site_figure(SITE_CONFIGS["WB02"])

## B. Lower Lang Suan River (WB07; 2025-01-20)

In [ ]:
# Cell B: Notebook 04 river-bend crop, aspect 1.60.
result_WB07 = create_site_figure(SITE_CONFIGS["WB07"])

## C. Pak Phanang River (WB05; 2025-07-04)

In [ ]:
# Cell C: Notebook 04 largest river-mouth component crop, aspect 1.60.
result_WB05 = create_site_figure(SITE_CONFIGS["WB05"])

In [ ]:
# Combined audit after all three site cells have run.
required_results = [result_WB02, result_WB07, result_WB05]
combined_summary = pd.concat(
    [result["summary"] for result in required_results], ignore_index=True
)
combined_summary.to_csv(
    TABLE_DIR / "figure1_three_sites_spatial_summary.csv",
    index=False, encoding="utf-8-sig",
)

audit_rows = []
for result in required_results:
    for output_type in ["prediction_path", "png", "pdf", "tif"]:
        path = Path(result[output_type])
        audit_rows.append({
            "map_id": result["site"]["map_id"],
            "output_type": output_type,
            "exists": path.exists(),
            "size_mb": path.stat().st_size / 1024**2 if path.exists() else np.nan,
            "path": str(path),
        })
    with rasterio.open(result["prediction_path"]) as source:
        if source.count != len(OUTPUT_BANDS) or list(source.descriptions) != OUTPUT_BANDS:
            raise RuntimeError(f"Prediction raster contract failed: {result['prediction_path']}")

output_audit = pd.DataFrame(audit_rows)
display(combined_summary.round(4))
display(output_audit.round(3))
if not output_audit["exists"].all():
    raise RuntimeError("At least one expected output is missing.")
print("THREE-SITE FIGURE 1 PIPELINE: COMPLETE")

In [ ]:
# Optional: choose your own private destination and uncomment to persist results.
# import shutil
# destination = Path("/content/drive/MyDrive/wqr_my_results")
# shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)
print("Results for this run:", OUTPUT_ROOT)
